# The Void and the Flux Field

**"The substrate of possibility and the carrier of energy."**

This notebook explores the two foundational concepts in FTD:

1. **The Void** (state = 0) - The dispositional substrate
2. **The Flux Field** J(v,t) - The continuous energy density

---

## Key Concepts

| Entity | Type | Role |
|--------|------|------|
| Void | Discrete (s=0) | Substrate awaiting activation |
| Flux | Continuous (R³) | Energy density and wave medium |
| Density | Scalar |J| | Manifestation potential |

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

repo_root = os.path.abspath("../../")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation, waves, forces
from ternary_matrix.config import CONSTANTS

# Configure for wave propagation study
CONSTANTS.C = 0.5
CONSTANTS.KB = 10.0  # High threshold - prevent manifestation
CONSTANTS.DECAY_RATE = 0.0
CONSTANTS.DAMPING = 0.01

print("Configuration loaded.")

## 1. The Discrete Wave Equation

The flux field evolves according to the **discrete wave equation**:

$$\frac{\partial^2 J}{\partial t^2} = C^2 \nabla^2 J$$

Where $\nabla^2$ is the **discrete Laplacian** over the 6-connected neighborhood:

$$\nabla^2 f(v) = \sum_{u \in N_6(v)} f(u) - 6f(v)$$

Let's visualize how this works:

In [ ]:
# Visualize the 6-connected neighborhood (N6)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Central voxel
ax.scatter([0], [0], [0], c='red', s=200, label='Central Voxel v', marker='o')

# 6 face-sharing neighbors
neighbors = [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]
for n in neighbors:
    ax.scatter([n[0]], [n[1]], [n[2]], c='blue', s=100, marker='s')
    ax.plot([0, n[0]], [0, n[1]], [0, n[2]], 'b--', alpha=0.5)

ax.scatter([], [], [], c='blue', s=100, marker='s', label='N₆ Neighbors')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('The 6-Connected Neighborhood (N₆) for Laplacian', fontsize=14)
ax.legend()
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_zlim(-1.5, 1.5)
plt.tight_layout()
plt.show()

## 2. Wave Propagation Experiment

Let's inject a flux pulse and watch it propagate as a spherical wave.

In [ ]:
# Create universe with central pulse
universe = Universe(size=64)
center = universe.size // 2

# Single-point pulse
universe.flux[center, center, center, :] = [5.0, 0.0, 0.0]  # X-directed pulse

print(f"Pulse injected at ({center}, {center}, {center})")
print(f"Initial flux: {universe.flux[center, center, center]}")

In [ ]:
# Capture wave propagation snapshots
snapshots = []
times = [0, 5, 10, 15, 20, 30, 40, 50]

for t in range(max(times) + 1):
    if t in times:
        # Get 2D slice through center
        flux_mag = np.linalg.norm(universe.flux[:, :, center, :], axis=-1).copy()
        snapshots.append((t, flux_mag))
    
    # Only propagate waves - no manifestation
    waves.propagate_flux(universe)
    forces.calculate_density(universe)

print(f"Captured {len(snapshots)} snapshots")

In [ ]:
# Display wave evolution
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Find global max for consistent colorscale
vmax = max(s[1].max() for s in snapshots) * 0.5  # Scale down for visibility

for ax, (t, flux_mag) in zip(axes, snapshots):
    im = ax.imshow(flux_mag, cmap='viridis', origin='lower', vmin=0, vmax=vmax)
    ax.set_title(f't = {t}', fontsize=12)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    # Add circle showing expected wavefront at r = C*t
    if t > 0:
        radius = CONSTANTS.C * t
        circle = plt.Circle((center, center), radius, fill=False, color='white', linestyle='--', linewidth=1)
        ax.add_patch(circle)

plt.suptitle('Wave Propagation: Flux Pulse Expanding at Speed C', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes, shrink=0.6, label='|J|')
plt.tight_layout()
plt.show()

## 3. Wave Speed Measurement

Let's verify that waves propagate at speed C by measuring the wavefront position.

In [ ]:
# Fresh universe for measurement
universe = Universe(size=64)
center = universe.size // 2
universe.flux[center, center, center, :] = [5.0, 0.0, 0.0]

# Track flux along a radial line (x-axis through center)
radial_profiles = []
measurement_times = list(range(0, 60, 2))

for t in range(max(measurement_times) + 1):
    if t in measurement_times:
        # Get flux magnitude along x-axis at y=center, z=center
        profile = np.linalg.norm(universe.flux[:, center, center, :], axis=-1).copy()
        radial_profiles.append((t, profile))
    
    waves.propagate_flux(universe)
    forces.calculate_density(universe)

In [ ]:
# Plot radial profiles as waterfall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Waterfall plot
x_axis = np.arange(universe.size) - center  # Distance from center

for i, (t, profile) in enumerate(radial_profiles[::3]):
    offset = i * 0.3
    axes[0].plot(x_axis, profile + offset, alpha=0.7, label=f't={t}')

axes[0].set_xlabel('Distance from center', fontsize=12)
axes[0].set_ylabel('|J| (offset for clarity)', fontsize=12)
axes[0].set_title('Radial Flux Profiles (Waterfall)', fontsize=12)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)

# Speed measurement: find peak positions
peak_positions = []
peak_times = []

for t, profile in radial_profiles:
    if t > 5:  # After initial spread
        # Find rightward peak
        right_half = profile[center:]
        if right_half.max() > 0.01:
            peak_idx = np.argmax(right_half)
            peak_positions.append(peak_idx)
            peak_times.append(t)

# Fit line to get speed
if len(peak_positions) > 3:
    coeffs = np.polyfit(peak_times, peak_positions, 1)
    measured_speed = coeffs[0]
    
    axes[1].scatter(peak_times, peak_positions, c='blue', s=50, label='Measured peaks')
    fit_line = np.polyval(coeffs, peak_times)
    axes[1].plot(peak_times, fit_line, 'r--', linewidth=2, label=f'Fit: v = {measured_speed:.3f}')
    
    # Theoretical line
    theoretical = [CONSTANTS.C * t for t in peak_times]
    axes[1].plot(peak_times, theoretical, 'g:', linewidth=2, label=f'Theoretical: C = {CONSTANTS.C}')
    
    axes[1].set_xlabel('Time (ticks)', fontsize=12)
    axes[1].set_ylabel('Peak distance from center', fontsize=12)
    axes[1].set_title('Wave Speed Measurement', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    print(f"Measured wave speed: {measured_speed:.4f}")
    print(f"Configured speed C: {CONSTANTS.C}")
    print(f"Error: {abs(measured_speed - CONSTANTS.C)/CONSTANTS.C * 100:.2f}%")

plt.tight_layout()
plt.show()

## 4. Two-Source Interference

When two flux sources are present, their waves **interfere** through vector addition.

This is the basis for quantum-like phenomena in FTD.

In [ ]:
# Two-source setup
universe = Universe(size=64)
center = universe.size // 2

# Two sources separated by 10 voxels
source1 = (center - 5, center, center)
source2 = (center + 5, center, center)

# Same phase (coherent sources)
universe.flux[source1[0], source1[1], source1[2], :] = [3.0, 0.0, 0.0]
universe.flux[source2[0], source2[1], source2[2], :] = [3.0, 0.0, 0.0]

print(f"Two coherent sources at x = {source1[0]} and x = {source2[0]}")

In [ ]:
# Capture interference pattern evolution
interference_snaps = []
times = [0, 10, 20, 30, 40, 50]

for t in range(max(times) + 1):
    if t in times:
        flux_mag = np.linalg.norm(universe.flux[:, :, center, :], axis=-1).copy()
        interference_snaps.append((t, flux_mag))
    
    waves.propagate_flux(universe)
    forces.calculate_density(universe)

In [ ]:
# Display interference evolution
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

vmax = max(s[1].max() for s in interference_snaps) * 0.3

for ax, (t, flux_mag) in zip(axes, interference_snaps):
    im = ax.imshow(flux_mag, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
    ax.set_title(f't = {t}', fontsize=12)
    
    # Mark source positions
    ax.scatter([source1[0], source2[0]], [source1[1], source2[1]], 
               c='yellow', s=50, marker='*', edgecolors='black')

plt.suptitle('Two-Source Interference: Constructive and Destructive Patterns', 
             fontsize=14, fontweight='bold')
fig.colorbar(im, ax=axes, shrink=0.6, label='|J|')
plt.tight_layout()
plt.show()

## 5. Visualizing Flux Components

The flux J is a **3D vector** with components (Jx, Jy, Jz). Let's visualize each component separately.

In [ ]:
# Create a more complex flux configuration
universe = Universe(size=48)
center = universe.size // 2

# Rotating flux pattern (vortex-like)
for x in range(universe.size):
    for y in range(universe.size):
        dx = x - center
        dy = y - center
        r = np.sqrt(dx**2 + dy**2) + 1e-10
        
        if r < 15:
            # Tangential flux (perpendicular to radial)
            magnitude = 2.0 * np.exp(-r**2 / 50)
            universe.flux[x, y, center, 0] = -magnitude * dy / r  # Jx
            universe.flux[x, y, center, 1] = magnitude * dx / r   # Jy
            universe.flux[x, y, center, 2] = 0  # Jz

print("Vortex flux pattern created.")

In [ ]:
# Visualize flux components
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

z = center
Jx = universe.flux[:, :, z, 0]
Jy = universe.flux[:, :, z, 1]
Jz = universe.flux[:, :, z, 2]
Jmag = np.sqrt(Jx**2 + Jy**2 + Jz**2)

components = [
    (Jx, 'Jx (X-component)', 'RdBu_r'),
    (Jy, 'Jy (Y-component)', 'RdBu_r'),
    (Jmag, '|J| (Magnitude)', 'inferno'),
]

for ax, (data, title, cmap) in zip(axes.flatten()[:3], components):
    vmax = abs(data).max()
    if 'RdBu' in cmap:
        im = ax.imshow(data, cmap=cmap, origin='lower', vmin=-vmax, vmax=vmax)
    else:
        im = ax.imshow(data, cmap=cmap, origin='lower')
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax)

# Vector field plot
ax = axes[1, 1]
step = 2
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Jx_sub = Jx[::step, ::step]
Jy_sub = Jy[::step, ::step]

ax.quiver(X, Y, Jx_sub, Jy_sub, np.sqrt(Jx_sub**2 + Jy_sub**2), cmap='viridis')
ax.set_title('Vector Field (Jx, Jy)', fontsize=12)
ax.set_xlim(0, universe.size)
ax.set_ylim(0, universe.size)
ax.set_aspect('equal')

plt.suptitle('Flux Field Components: Vortex Configuration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Energy Conservation

In a closed system with no damping, total flux energy should be conserved.

Energy ~ ∫|J|² dV (integral of flux squared)

In [ ]:
# Test energy conservation
CONSTANTS.DAMPING = 0.0  # No damping
CONSTANTS.DECAY_RATE = 0.0  # No decay

universe = Universe(size=32)
center = universe.size // 2

# Random flux injection
universe.flux[center-3:center+3, center-3:center+3, center-3:center+3, :] = np.random.randn(6, 6, 6, 3)

# Track total energy
energy_history = []

for t in range(100):
    total_energy = np.sum(universe.flux**2)
    energy_history.append(total_energy)
    
    waves.propagate_flux(universe)

# Plot energy over time
plt.figure(figsize=(10, 5))
plt.plot(energy_history, 'b-', linewidth=2)
plt.axhline(energy_history[0], color='r', linestyle='--', label=f'Initial E = {energy_history[0]:.2f}')
plt.xlabel('Time (ticks)', fontsize=12)
plt.ylabel('Total Energy (Σ|J|²)', fontsize=12)
plt.title('Energy Conservation Test (No Damping)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Calculate drift
drift = (energy_history[-1] - energy_history[0]) / energy_history[0] * 100
print(f"Initial energy: {energy_history[0]:.4f}")
print(f"Final energy: {energy_history[-1]:.4f}")
print(f"Relative drift: {drift:.4f}%")

plt.show()

## 7. The Role of Damping

In the real framework, **damping** provides energy dissipation (entropy). Let's compare.

In [ ]:
damping_values = [0.0, 0.01, 0.05, 0.1]
results = {}

for damp in damping_values:
    CONSTANTS.DAMPING = damp
    
    universe = Universe(size=32)
    center = universe.size // 2
    universe.flux[center-3:center+3, center-3:center+3, center-3:center+3, :] = np.random.randn(6, 6, 6, 3) * 3
    
    energy = []
    for t in range(100):
        energy.append(np.sum(universe.flux**2))
        waves.propagate_flux(universe)
    
    results[damp] = energy

# Plot comparison
plt.figure(figsize=(10, 5))
for damp, energy in results.items():
    # Normalize to initial
    energy_norm = np.array(energy) / energy[0]
    plt.plot(energy_norm, linewidth=2, label=f'Damping = {damp}')

plt.xlabel('Time (ticks)', fontsize=12)
plt.ylabel('Normalized Energy (E/E₀)', fontsize=12)
plt.title('Effect of Damping on Energy Dissipation', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.show()

## 8. Summary: The Void and Flux

### Key Takeaways

1. **The Void** is not "nothing" - it's a dispositional substrate
2. **The Flux** is a continuous 3D vector field carrying energy
3. **Waves** propagate at speed C via the discrete wave equation
4. **Interference** occurs through vector addition
5. **Energy** is conserved (without damping)
6. **Damping** provides the arrow of time (entropy)

### The Two-Layer Ontology

| Layer | Nature | Role |
|-------|--------|------|
| Flux (J) | Continuous | Energy carrier, wave medium |
| State (s) | Discrete | Manifested matter/antimatter |

**Next**: See `02_manifestation.ipynb` for how flux becomes matter.